In [ ]:
!pip uninstall -y unsloth unsloth_zoo
!pip install --no-cache-dir -U \
  git+https://github.com/unslothai/unsloth.git \
  git+https://github.com/unslothai/unsloth-zoo.git \
  trl peft accelerate bitsandbytes datasets


In [ ]:
import os
os.environ['MODELSCOPE_CACHE'] = './model_scope_cache'
os.environ['HF_ENDPOINT'] = 'https://hf-mirror.com'

import torch
from modelscope import snapshot_download
from unsloth import FastLanguageModel

# 1. 明确使用魔搭下载原版模型到本地缓存
# 注意：魔搭上 Qwen 官方的组织名通常是小写的 qwen
model_id = "qwen/Qwen3-0.6B"
print(f"正在从魔搭下载模型 {model_id}，请稍候...")
local_model_path = snapshot_download(model_id)
print(f"模型已成功下载至本地路径: {local_model_path}")

# 2. 从【本地路径】加载模型
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = local_model_path,  # <--- 关键：这里传入上一步返回的本地绝对路径
    max_seq_length = 2048,
    dtype = None,                   # 自动推断
    load_in_4bit = False,            # 开启 4-bit，这会在本地加载时即时量化，不依赖 HF
)

# 3. 正常配置 LoRA 参数
model = FastLanguageModel.get_peft_model(
    model,
    r = 16,
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj"],
    lora_alpha = 16,
    lora_dropout = 0,
    bias = "none",
    use_gradient_checkpointing = "unsloth",
    random_state = 3407,
)

print("模型加载与 LoRA 配置成功！不再会有网络超时报错了。")

In [ ]:
import json
from datasets import load_dataset

# 1. 改回标准 Alpaca 模板 (使用 ### Response:)
alpaca_prompt = """Below is an instruction that describes a task. Write a response that appropriately completes the request.

### Instruction:
{}

### Input:
{}

### Response:
{}"""

EOS_TOKEN = tokenizer.eos_token

def formatting_prompts_func(examples):
    instructions = examples["instruction"]
    inputs       = examples["input"]
    outputs      = examples["output"]
    texts = []
    for instruction, input, output in zip(instructions, inputs, outputs):
        # 【核心修改】清洗数据：如果 output 是字典，去掉所有值为 None 的键
        if isinstance(output, dict):
            # 这一行是关键：只保留值不为 None 的字段
            clean_output = {k: v for k, v in output.items() if v is not None}
            output_str = json.dumps(clean_output, ensure_ascii=False)
        else:
            output_str = str(output)

        text = alpaca_prompt.format(instruction, input, output_str) + EOS_TOKEN
        texts.append(text)
    return { "text" : texts, }

# 加载数据
dataset = load_dataset("json", data_files="train.jsonl", split="train")
dataset = dataset.map(formatting_prompts_func, batched = True)

# 3. 【新增】打印一条处理后的数据，检查格式是否正确！
print("检查第一条训练数据格式：")
print(dataset["text"][0])
# 务必确认这里看到的是 {"intent": ...} (双引号)，而不是 {'intent': ...} (单引号)

In [ ]:
from trl import SFTTrainer
from transformers import TrainingArguments
from unsloth import is_bfloat16_supported

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset,
    dataset_text_field = "text",
    max_seq_length = 2048,
    dataset_num_proc = 2,
    packing = False,
    args = TrainingArguments(
        per_device_train_batch_size = 4, # 显存允许的话，越小更新越频繁
        gradient_accumulation_steps = 4,
        warmup_steps = 5,
        # max_steps = 60,  <-- 删掉这个
        num_train_epochs = 15, # 【修改】直接指定跑 15 轮，确保学会格式
        learning_rate = 2e-4,
        fp16 = not is_bfloat16_supported(),
        bf16 = is_bfloat16_supported(),
        logging_steps = 1,
        optim = "adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type = "linear",
        seed = 3407,
        output_dir = "outputs",
        # 不弹出wandb选择框
        report_to = "none",
    ),
)

trainer_stats = trainer.train()

In [ ]:
from datasets import load_dataset

# Load the train.jsonl dataset for testing
test_dataset = load_dataset("json", data_files="train.jsonl", split="train")

print("✅ Test dataset loaded successfully!")
print(f"Number of examples in test dataset: {len(test_dataset)}")
print("First example:")
print(test_dataset[0])

In [ ]:
FastLanguageModel.for_inference(model) # Set model to inference mode

test_results = []

print("✅ Model set to inference mode and `test_results` list initialized.")

In [ ]:
import json

for i, example in enumerate(test_dataset):
    instruction = example["instruction"]
    input_text = example["input"]
    expected_output_str = example["output"]

    # Construct the prompt for inference
    prompt = alpaca_prompt.format(
        instruction,
        input_text,
        "", # No output provided during inference
    )

    # Tokenize the prompt
    inputs = tokenizer([prompt], return_tensors="pt").to("cuda")

    # Generate output from the model
    outputs = model.generate(**inputs, max_new_tokens=128, use_cache=True)

    # Decode the generated output, skipping the prompt part
    decoded_output_full = tokenizer.decode(outputs[0], skip_special_tokens=True)

    # Extract only the generated response (after the ### Response: part)
    response_start_index = decoded_output_full.find("### Response:")
    if response_start_index != -1:
        generated_output_str = decoded_output_full[response_start_index + len("### Response:"):].strip()
    else:
        generated_output_str = decoded_output_full.strip() # Fallback if ### Response: isn't found

    # Store results
    test_results.append({
        "instruction": instruction,
        "input": input_text,
        "expected_output": expected_output_str,
        "generated_output": generated_output_str,
    })

print(f"✅ Generated responses for {len(test_results)} examples and stored them in `test_results`.")
# print(test_results[0]) # Print first result to check format

In [ ]:
import json

# --- 1. 初始化统计变量 ---
total_cases = len(test_results)
match_count = 0
failed_items = []

# --- 2. 核心处理与评估循环 ---
for i, result in enumerate(test_results):
    def safe_get_dict(item):
        """处理嵌套字符串、转义斜杠并剔除 None 字段"""
        if item is None: return {}
        # 如果是字符串，尝试解析回字典
        if isinstance(item, str):
            try:
                data = json.loads(item)
                # 处理双重序列化问题
                if isinstance(data, str): 
                    data = json.loads(data)
                item = data
            except:
                return {"raw_text": item}
        
        # 统一为字典后，剔除所有 None 值的键
        if isinstance(item, dict):
            return {k: v for k, v in item.items() if v is not None}
        return {}

    def normalize_values(d):
        """统一数据类型，防止 2.0 vs '2.0' 导致判定失败"""
        # 统一数值字段为浮点数
        for k in ["value", "percent"]:
            if k in d:
                try: d[k] = float(d[k])
                except: pass
        # 统一通道字段为整数
        if "channel" in d:
            try: d["channel"] = int(float(d["channel"]))
            except: pass
        # 针对本项目：统一 channels 为列表格式
        if "channels" in d and not isinstance(d["channels"], list):
            d["channels"] = [d["channels"]]
        return d

    # 执行清洗与规范化
    expected_dict = normalize_values(safe_get_dict(result.get("expected_output")))
    generated_dict = normalize_values(safe_get_dict(result.get("generated_output")))
    
    # 逻辑比较 (字典比较不受键的顺序、空格或引号影响)
    is_match = (expected_dict == generated_dict)
    
    # 更新结果状态
    result["match_status"] = is_match
    if is_match:
        match_count += 1
    else:
        # 记录失败项
        failed_items.append({
            "index": i + 1,
            "input": result.get("input"),
            "expected": json.dumps(expected_dict, ensure_ascii=False),
            "generated": json.dumps(generated_dict, ensure_ascii=False)
        })

    # 为了美观打印，存入带双引号的 JSON 字符串
    result["expected_display"] = json.dumps(expected_dict, ensure_ascii=False)
    result["generated_display"] = json.dumps(generated_dict, ensure_ascii=False)

# --- 3. 打印详细统计报告 ---
accuracy = (match_count / total_cases) * 100 if total_cases > 0 else 0

print("\n" + "="*60)
print(f"🚀 示波器指令解析测试报告")
print(f"总测试用例数: {total_cases}")
print(f"匹配成功总数: {match_count}")
print(f"匹配成功百分比: {accuracy:.2f}%")
print("="*60)

# --- 4. 单独列出失败的项目 ---
if failed_items:
    print(f"\n❌ 检测到 {len(failed_items)} 个失败的项目：")
    for item in failed_items:
        print(f"\n[案例 {item['index']}]")
        print(f"  指令输入: {item['input']}")
        print(f"  预期输出: {item['expected']}")
        print(f"  模型输出: {item['generated']}")
        print("-" * 30)
else:
    print("\n🎉 完美！所有指令解析结果均与预期完全匹配。")

In [ ]:
model.save_pretrained("qwen-0.6b-lora-only")
tokenizer.save_pretrained("qwen-0.6b-lora-only")

In [ ]:
import os
import torch
from modelscope import snapshot_download
from unsloth import FastLanguageModel

# 1. 重新设定缓存环境变量，确保能找到底座模型
os.environ['MODELSCOPE_CACHE'] = './model_scope_cache'

print("正在干净的环境中以 16-bit 模式加载模型...")
# 2. 直接读取你刚才保存的 LoRA 文件夹，它会自动找到关联的底座
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "qwen-0.6b-lora-only", # <--- 填入你保存的 LoRA 适配器路径
    max_seq_length = 2048,
    dtype = torch.float16,  # 强制 16-bit
    load_in_4bit = False,   # 彻底关闭 4-bit
)

print("加载成功，开始执行 16-bit 合并...")
# 3. 合并并导出完整的 16-bit 模型
model.save_pretrained_merged(
    "qwen-0.6b-lora-merged-16bit", 
    tokenizer, 
    save_method = "merged_16bit"
)

print("🎉 大功告成！合并后的完整模型已保存在 qwen-0.6b-lora-merged-16bit 目录。")

In [ ]:
!zip -r qwen-0.6b-lora-merged.zip qwen-0.6b-lora-merged-16bit/

## Load Merged Model

### Subtask:
Load the previously saved merged model from the 'qwen-0.6b-lora-merged' directory. Ensure the model is set up for inference.


In [ ]:
from unsloth import FastLanguageModel

# Define the path to the merged model
model_path = "qwen-0.6b-lora-merged-16bit"

# Load the merged model and tokenizer
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = model_path,
    load_in_4bit = False, # Load in 4-bit precision for efficiency
)

# Set the model to inference mode
FastLanguageModel.for_inference(model)

print(f"✅ Model '{model_path}' loaded and set for inference.")

## Perform Inference with Merged Model

### Subtask:
Iterate through the `test_dataset`, construct prompts using `alpaca_prompt`, and generate responses using the loaded merged model. Store the generated outputs along with instructions, inputs, and expected outputs in a list.


**Reasoning**:
To perform inference with the merged model as per the subtask, I will iterate through the `test_dataset`, construct prompts, generate responses, and store them in `inference_results`.



In [ ]:
inference_results = []

for i, example in enumerate(test_dataset):
    instruction = example["instruction"]
    input_text = example["input"]
    expected_output_str = example["output"]

    # Construct the prompt for inference
    prompt = alpaca_prompt.format(
        instruction,
        input_text,
        "", # No output provided during inference
    )

    # Tokenize the prompt
    inputs = tokenizer([prompt], return_tensors="pt").to("cuda")

    # Generate output from the model
    outputs = model.generate(**inputs, max_new_tokens=128, use_cache=True)

    # Decode the generated output, skipping the prompt part
    decoded_output_full = tokenizer.decode(outputs[0], skip_special_tokens=True)

    # Extract only the generated response (after the ### Response: part)
    response_start_index = decoded_output_full.find("### Response:")
    if response_start_index != -1:
        generated_output_str = decoded_output_full[response_start_index + len("### Response:"):].strip()
    else:
        generated_output_str = decoded_output_full.strip() # Fallback if ### Response: isn't found

    # Store results
    inference_results.append({
        "instruction": instruction,
        "input": input_text,
        "expected_output": expected_output_str,
        "generated_output": generated_output_str,
    })

print(f"✅ Generated responses for {len(inference_results)} examples and stored them in `inference_results`.")

In [ ]:
import json

# --- 1. 初始化统计变量 ---
total_cases = len(inference_results)
match_count = 0
failed_items = []

# --- 2. 核心处理与评估循环 ---
for i, result in enumerate(inference_results):
    def safe_get_dict(item):
        """处理嵌套字符串、转义斜杠并剔除 None 字段"""
        if item is None: return {}
        # 如果是字符串，尝试解析回字典
        if isinstance(item, str):
            try:
                data = json.loads(item)
                # 处理双重序列化问题
                if isinstance(data, str): 
                    data = json.loads(data)
                item = data
            except:
                return {"raw_text": item}
        
        # 统一为字典后，剔除所有 None 值的键
        if isinstance(item, dict):
            return {k: v for k, v in item.items() if v is not None}
        return {}

    def normalize_values(d):
        """统一数据类型，防止 2.0 vs '2.0' 导致判定失败"""
        # 统一数值字段为浮点数
        for k in ["value", "percent"]:
            if k in d:
                try: d[k] = float(d[k])
                except: pass
        # 统一通道字段为整数
        if "channel" in d:
            try: d["channel"] = int(float(d["channel"]))
            except: pass
        # 针对本项目：统一 channels 为列表格式
        if "channels" in d and not isinstance(d["channels"], list):
            d["channels"] = [d["channels"]]
        return d

    # 执行清洗与规范化
    expected_dict = normalize_values(safe_get_dict(result.get("expected_output")))
    generated_dict = normalize_values(safe_get_dict(result.get("generated_output")))
    
    # 逻辑比较 (字典比较不受键的顺序、空格或引号影响)
    is_match = (expected_dict == generated_dict)
    
    # 更新结果状态
    result["match_status"] = is_match
    if is_match:
        match_count += 1
    else:
        # 记录失败项
        failed_items.append({
            "index": i + 1,
            "input": result.get("input"),
            "expected": json.dumps(expected_dict, ensure_ascii=False),
            "generated": json.dumps(generated_dict, ensure_ascii=False)
        })

    # 为了美观打印，存入带双引号的 JSON 字符串
    result["expected_display"] = json.dumps(expected_dict, ensure_ascii=False)
    result["generated_display"] = json.dumps(generated_dict, ensure_ascii=False)

# --- 3. 打印详细统计报告 ---
accuracy = (match_count / total_cases) * 100 if total_cases > 0 else 0

print("\n" + "="*60)
print(f"🚀 示波器指令解析测试报告")
print(f"总测试用例数: {total_cases}")
print(f"匹配成功总数: {match_count}")
print(f"匹配成功百分比: {accuracy:.2f}%")
print("="*60)

# --- 4. 单独列出失败的项目 ---
if failed_items:
    print(f"\n❌ 检测到 {len(failed_items)} 个失败的项目：")
    for item in failed_items:
        print(f"\n[案例 {item['index']}]")
        print(f"  指令输入: {item['input']}")
        print(f"  预期输出: {item['expected']}")
        print(f"  模型输出: {item['generated']}")
        print("-" * 30)
else:
    print("\n🎉 完美！所有指令解析结果均与预期完全匹配。")